In [ ]:
#bash
#!pip install -q flwr[simulation] flwr_datasets[vision] torch torchvision matplotlib tensorflow scikit-learn 

#zsh
#!pip install -q flwr\[simulation\] flwr_datasets\[vision\] torch torchvision matplotlib tensorflow scikit-learn 

In [ ]:
# Import necessary libraries
import flwr as fl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import logging
import random
import time

from sklearn.metrics import precision_score, f1_score, confusion_matrix, balanced_accuracy_score
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

# Set up logging
logging.basicConfig(level=logging.INFO)

DEVICE = torch.device("cuda")  # Try "cuda" to train on GPU
#DEVICE = torch.device("cuda")  # Try "cuda" to train on GPU
print(
    f"Training on {DEVICE} using PyTorch {torch.__version__} and Flower {fl.__version__}"
)

In [ ]:
PATH = '../datasets/pamap_balanced.csv'
SEQUENCE_LENGTH = 60
NUM_CLASSES = 6
BATCH_SIZE = 64
NUM_EPOCHS = 80
NUM_ROUNDS = 40
LEARNING_RATE = 0.001
NUM_CLIENTS = 5
TRAINING_PERCENTAGE = 80

# Define client's resources
CLIENT_RESOURCES = {'num_cpus': 2.0}

ID_COLUMN = 'subject_id'
ACTIVITY_COLUMN = 'activity_id'
FEATURE_COLUMNS = [
                'hand_acc_16g_x', 'hand_acc_16g_y', 'hand_acc_16g_z',
                'hand_acc_6g_x', 'hand_acc_6g_y', 'hand_acc_6g_z',	
                'hand_gyro_x', 'hand_gyro_y', 'hand_gyro_z',	
                'hand_mag_x', 'hand_mag_y', 'hand_mag_z'
            ]

In [ ]:
def load_pamap_data(file_path, num_users=10, percentage_training=80):
    data = []
    labels = []
    users = []
    df = pd.read_csv(file_path)
    
    for index, row in df.iterrows():
        user = row[ID_COLUMN]
        activity = row[ACTIVITY_COLUMN]
        features = row[FEATURE_COLUMNS].values
        
        data.append(features)
        labels.append(activity)
        users.append(user)
    
    # Convert to numpy arrays
    data = np.array(data) #remove column names
    labels = np.array(labels)
    users = np.array(users)

    # Filter users
    unique_users = np.unique(users)
    if num_users > len(unique_users):
        num_users = len(unique_users)  # Ensure num_users does not exceed available unique users
    selected_users = set(random.sample(list(unique_users), num_users))
    
    selected_indices = [i for i, user in enumerate(users) if user in selected_users]
    
    filtered_data = data[selected_indices]
    filtered_labels = labels[selected_indices]
    filtered_users = users[selected_indices]
    
    # Split users into training and testing sets
    unique_selected_users = np.unique(filtered_users)
    num_train_users = int(percentage_training / 100 * len(unique_selected_users))
    
    train_users = set(random.sample(list(unique_selected_users), num_train_users))
    test_users = set(unique_selected_users) - train_users
    
    train_indices = [i for i, user in enumerate(filtered_users) if user in train_users]
    test_indices = [i for i, user in enumerate(filtered_users) if user in test_users]
    
    x_train = filtered_data[train_indices]
    y_train = filtered_labels[train_indices]
    x_test = filtered_data[test_indices]
    y_test = filtered_labels[test_indices]
    
    return x_train, y_train, x_test, y_test

In [ ]:
X_train, y_train, X_test, y_test = load_pamap_data(PATH, NUM_CLIENTS, TRAINING_PERCENTAGE)
print(f'X_train shape: {X_train.shape}, y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}, y_test shape: {y_test.shape}')

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Encode labels to integers
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.fit_transform(y_test)

# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

# Reshape data into sequences
def create_sequences(data, labels, sequence_length):
    sequences = []
    seq_labels = []
    for i in range(len(data) - sequence_length):
        seq = data[i:i+sequence_length]
        label = labels[i+sequence_length-1]
        sequences.append(seq)
        seq_labels.append(label)
    
    return np.array(sequences), np.array(seq_labels)

X_train, y_train = create_sequences(X_train, y_train, SEQUENCE_LENGTH)
X_test, y_test = create_sequences(X_test, y_test, SEQUENCE_LENGTH)

X_train = np.transpose(X_train, (0, 2, 1))
X_test = np.transpose(X_test, (0, 2, 1))

# Convert data to PyTorch tensors
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {X_test.shape}, Testing labels shape: {y_test.shape}')

In [ ]:
class PEN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(PEN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv1d(in_channels=input_size, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)

        self.conv2 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(128)

        self.conv3 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=2, stride=1, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        
        # LSTM layers
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=2, batch_first=True)
        
        # Fully connected layers
        self.fc = nn.Linear(hidden_size + 128, num_classes)

    def forward(self, x):
        # Convolutional path
        x_cnn = self.conv1(x)
        x_cnn = self.bn1(x_cnn)
        x_cnn = self.relu(x_cnn)
        x_cnn = self.pool(x_cnn)

        x_cnn = self.conv2(x_cnn)
        x_cnn = self.bn2(x_cnn)
        x_cnn = self.relu(x_cnn)
        x_cnn = self.pool(x_cnn)

        x_cnn = self.conv3(x_cnn)
        x_cnn = self.bn3(x_cnn)
        x_cnn = self.relu(x_cnn)
        x_cnn = self.pool(x_cnn)
        
        x_cnn = x_cnn.mean(dim=-1)  # Global Average Pooling

        # LSTM path
        x_lstm, _ = self.lstm(x.transpose(1, 2))  # Transpose to match LSTM input shape
        x_lstm = x_lstm[:, -1, :]  # Take the last output of the sequence

        # Concatenate CNN and LSTM paths
        x = torch.cat((x_cnn, x_lstm), dim=1)

        # Fully connected layer
        x = self.fc(x)
        return x

In [ ]:
def compute_metrics(loader, model):
    all_preds = []
    all_labels = []
    for x_batch, y_batch in loader:
        data, labels = x_batch.to(DEVICE), y_batch.to(DEVICE)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy()) # Transform to numpy to add to the full list of predictions
        all_labels.extend(labels.cpu().numpy())
        
    balanced_acc = balanced_accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds, average='weighted') * 100
    precision = precision_score(all_labels, all_preds, average='weighted') * 100

    
    return balanced_acc, f1, precision

In [ ]:
def train(model, train_loader, test_loader, criterion, optimizer, num_epochs, verbose=False):
    model.train()
    total_training_time = 0
    times = []
    train_losses = []
    train_accuracies = []
    train_f1s = []
    train_precisions = []
    test_accuracies = []
    test_f1s = []
    test_precisions = []
    
    for epoch in range(num_epochs):
        start_epoch_time = time.time()
        total_loss = 0
        
    
        for x_batch, y_batch in train_loader:
            data, labels = x_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        end_epoch_time = time.time()
        epoch_time = end_epoch_time - start_epoch_time
        total_training_time += epoch_time
        
        train_acc, train_f1, train_precision = compute_metrics(train_loader, model)
        train_accuracies.append(train_acc)
        train_f1s.append(train_f1)
        train_precisions.append(train_precision)

        test_acc, test_f1, test_precision = compute_metrics(test_loader, model)
        test_accuracies.append(test_acc)
        test_f1s.append(test_f1)
        test_precisions.append(test_precision)
        
        times.append(epoch_time)
        train_losses.append(avg_loss)
        if(verbose):
            print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss:.4f}, Train. Accuracy: {train_acc:.2f},\
            \nTest Accuracy: {test_acc:.2f}, Time: {epoch_time:.2f} seconds')
    if(verbose):
        print(f'Total Training Time: {total_training_time:.2f} seconds')

    return train_losses, train_accuracies, train_f1s, train_precisions, test_accuracies, test_f1s, test_precisions

In [ ]:
# Define model, criterion, and optimizer
input_size = X_train.shape[1]
hidden_size = 128
num_classes = len(np.unique(y_train))

model = PEN(input_size, hidden_size, num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# Train the model
train_losses, train_accuracies, train_f1s, train_precisions, test_accuracies, test_f1s, test_precisions = train(model, train_loader, test_loader, criterion, optimizer, NUM_EPOCHS, verbose=True)

In [ ]:
def plot_metrics(train, test, title, metric):
    # summarize history for accuracy
    plt.plot(train)
    plt.plot(test)
    plt.title(title)
    plt.ylabel(metric)
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='best')
    plt.show()

In [ ]:
print(f'Final Accuracy: {test_accuracies[-1]:.2f}')
print(f'Final F1 Score: {test_f1s[-1]:.2f}')
print(f'Final Precision: {test_precisions[-1]:.2f}')

In [ ]:
plot_metrics(train_accuracies, test_accuracies, 'Accuracy over Epochs (PAMAP)', 'Balanced Accuracy')

In [ ]:
plot_metrics(train_f1s, test_f1s, 'F1 Score over Epochs (PAMAP)', 'F1 Score')

In [ ]:
plot_metrics(train_precisions, test_precisions, 'Precision over Epochs (PAMAP)', 'Precision')